In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os, sys, time
import geopandas as gpd

REPO = Path(os.getcwd()).parent
DATA_ROOT = REPO.parent.parent / 'Data' / 'replication_package'
PLOT_ROOT = DATA_ROOT.parent.parent / 'Plots' / 'Regional'

sys.path.insert(0, os.path.join(REPO, 'Code', 'tools'))
import inequality_analyzers as ia

In [15]:
# Load CBOS data
cbos = pd.read_csv(DATA_ROOT / "CBOS" / "CBOS_survey.csv", low_memory=False)

# Load LIS Voiv
lis_voiv = pd.read_csv(DATA_ROOT / "LIS" / "LIS_Voiv.csv")
# Load LIS Groups
lis_groups = pd.read_csv(DATA_ROOT / "LIS" / "LIS_Groups.csv")

In [16]:
# We create relevant columns
cbos["H_total_income"] = cbos["income_hh_t_imputed"]        # Total household income: income per household member multiplied by household size
cbos["P_income"] = cbos["income_p_imputed"]

# Regional aspect:
# - 1990-1998: old voivodeships (49 units) -> location_old_L
# - 1999-2.2017: new voivodeships (16 units) -> location_new_L
# - 3.2017-12.2017: macroregions (6 units) -> macroregion

# Relevant region columns: location_new_L, location_old_L (and _L their labels), macroregion
# Relevant demographic columns: age, sex_L, household_size, educ_1990_L old education division and educ_2000_L new
# Relevant technical columns: survey_year, survey_month, survey_date, survey_file
# Relevant geolocation columns: G_VOIV_500, G_VOIV_100, G_VOIV_100_500, G_MACRO_500, G_MACRO_100, G_MACRO_100_500

# Original weight column: weight
# Additional weights, their household versions and their normalized versions:
# - weight_VOIV (_h) (_NORM):               weight inside voivodeships (old, new or macroregions whatever is available)
# - weight_VOIV_500 (_h) (_NORM):           weight inside cities 500k+ and rest of voivodeships (old, new or macroregions whatever is available)
# - weight_VOIV_100 (_h) (_NORM):           weight inside cities 100k+ and rest of voivodeships (old, new or macroregions whatever is available)
# - weight_VOIV_100_500 (_h) (_NORM):       weight inside cities 100k+ and 500k+ and rest of voivodeships (old, new or macroregions whatever is available) 
# - weight_MACRO (_h) (_NORM):              weight inside macroregions (available from 1999 onwards)
# - weight_MACRO_500 (_h) (_NORM):          weight inside cities 500k+ and rest of macroregions (available from 1999 onwards)
# - weight_MACRO_100 (_h) (_NORM):          weight inside cities 100k+ and rest of macroregions (available from 1999 onwards)
# - weight_MACRO_100_500 (_h) (_NORM):      weight inside cities 100k+ and 500k+ and rest of macroregions (available from 1999 onwards)

# Deflators (for base year in [1986, 1990, 2017, 2023]):
# - deflator_YYYY: average yearly deflator, with base year in YYYY.
# - deflator_YYYY_m: monthly deflator, with base year in January of YYYY.


In [17]:
# Load plotting package
region_gdf = gpd.read_file(DATA_ROOT / 'geometry' / 'geometry.gpkg', layer='old_voivodeships')

In [18]:
# Initialize analyzers
# CBOS: microdata analyzer with 2017 PLN deflation
cbos_a = ia.CBOSAnalyzer(cbos, 
                         income_col='income_hh_imputed',
                         weight_col='weight_VOIV_NORM',
                         deflator_col='deflator_2017'
                         )

# LIS: pre-computed aggregates
lis_voiv_a = ia.LISAnalyzer(lis_voiv, income_type='pitotalnet', deflator_col='deflator_2017')
lis_groups_a = ia.LISAnalyzer(lis_groups, income_type='pitotalnet', deflator_col='deflator_2017')

# Quick summaries
print("CBOS:", cbos_a.summary())
print("LIS Voiv:", lis_voiv_a.summary())
print("LIS Groups:", lis_groups_a.summary())


CBOS: {'n_observations': 355337, 'year_range': (np.int64(1990), np.int64(2017)), 'n_years': 28, 'income_col': 'income_hh_imputed', 'weight_col': 'weight_VOIV_NORM', 'deflator_col': 'deflator_2017', 'income_non_null': np.int64(355337), 'weight_non_null': np.int64(355337)}
LIS Voiv: {'income_type': 'pitotalnet', 'n_regions': 71, 'year_range': (np.int64(1986), np.int64(2023)), 'n_metrics': 31, 'available_metrics': ['N_total', 'Nw_total', 'mean', 'median', 'p10', 'p25', 'p75', 'p90', 'p99', 'p90p10', 'p90p50', 'p50p10', 'gini', 'theil', 'palma', 'N_Bottom_50', 'Nw_Bottom_50', 'Bottom_50', 'N_P50_90', 'Nw_P50_90', 'P50_90', 'N_Top_10', 'Nw_Top_10', 'Top_10', 'N_Top_1', 'Nw_Top_1', 'Top_1', 'share_Bottom_50', 'share_P50_90', 'share_Top_10', 'share_Top_1'], 'deflator': 'deflator_2017'}
LIS Groups: {'income_type': 'pitotalnet', 'n_regions': 141, 'year_range': (np.int64(1999), np.int64(2023)), 'n_metrics': 31, 'available_metrics': ['N_total', 'Nw_total', 'mean', 'median', 'p10', 'p25', 'p75', '

In [7]:
cbos_a.regional_metrics(year = 1990, region_col='location_old_L', region_id_col='teryt_id_VOIV')

/Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/local_repo/LRDWI-Paper/Code/tools/inequality_analyzers.py:184: RuntimeWarning: divide by zero encountered in scalar divide
  frac = (cum_w[boundary] - 0.9 * total_w) / sw[boundary]
/Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/local_repo/LRDWI-Paper/Code/tools/inequality_analyzers.py:185: RuntimeWarning: invalid value encountered in scalar multiply
  t10_income += frac * weighted_inc[boundary]


,N_total,Nw_total,mean,median,p10,p25,p50,p75,p90,p99,...,Top_10,N_Top_10,Nw_Top_10,share_Top_1,Top_1,N_Top_1,Nw_Top_1,region,region_id,year
0,479,128.744053,427.817318,314.369800,157.184900,220.058860,314.369800,471.554700,817.361480,1886.218800,...,1242.085269,57,13.102451,0.083125,1983.140140,8,2.308691,bialskopodlaskie,9900000,1990
1,727,285.623999,553.615028,322.229045,165.044145,251.495840,322.229045,550.147150,1180.726914,3143.698000,...,2094.715012,68,28.789581,0.093335,3474.946369,6,4.247184,białostockie,9800000,1990
2,615,363.928795,655.770520,392.962250,157.184900,267.214330,392.962250,708.117974,1571.849000,3262.372599,...,2463.388334,88,36.789746,0.069468,4520.503766,9,3.667461,bielskie,9700000,1990
3,551,455.179960,759.609762,471.554700,235.777350,361.525270,471.554700,904.599099,1690.523599,3458.067800,...,2318.407867,70,45.661639,0.054134,3756.610678,7,4.982547,bydgoskie,9600000,1990
4,112,109.318176,917.890467,628.739600,157.184900,314.369800,628.739600,1297.561349,1690.523599,3065.891475,...,2353.264669,12,11.558921,0.068970,3139.153056,2,2.204601,chełmskie,9500000,1990
5,99,181.142478,935.200971,707.332050,211.376406,471.554700,707.332050,1297.561349,1690.523599,3183.378699,...,2357.166245,13,18.161005,0.044939,3296.561273,3,2.309336,ciechanowskie,9400000,1990
6,299,320.875271,907.795643,565.865640,314.369800,392.962250,565.865640,1297.561349,2122.782074,3262.372599,...,2640.879606,34,32.134815,0.048966,3392.262689,5,4.204661,częstochowskie,9300000,1990
7,139,210.334408,892.354670,550.147150,220.058860,314.369800,550.147150,1297.561349,2122.782074,3537.446174,...,2745.853431,18,21.374105,0.048250,3537.446175,1,2.560086,elbląskie,9200000,1990
8,569,586.930970,957.031635,708.117974,282.932820,392.962250,708.117974,1297.561349,2083.485849,3537.446174,...,2729.425453,87,60.321575,0.041483,3893.326844,9,5.985049,gdańskie,9100000,1990
9,202,202.460763,1010.745251,550.147150,314.369800,424.399230,550.147150,1297.561349,2083.485849,4952.110274,...,3438.344295,32,20.847630,0.085637,5072.557046,5,3.454748,gorzowskie,9000000,1990
